# Position distribution of the small, quiet events (-82 V, DCRC1pB trigger)

*Draft prepared with Claude Code for A. Villano, 2026-11-02.*

The position of an event on the face from the delays between the 20 % rise times of the channels (`xdel`, `ydel`, `phidel`, `afdel` of `positions-82v-dcrc1pb.ipynb`, after Fritts's `r76_analysis_position_new.c`), for the events that pass two cuts only:

* **`bstd` on all channels**: in every one of the six channels `bstd` (the baseline noise) is below that channel's dotted line of `bstd-distribution-450.ipynb`, twice the median of the random triggers (A 4.10, B 6.15, C 3.20, D 6.31, E 3.37, F 3.54 counts);
* **small**: **no** channel has a pulse height (`current_max`, counts above the baseline) larger than **5 %** of its saturated height (A 1277, B 1095, C 589, D 930, E 680, F 883 counts, from `saturation-over-time.ipynb`; 64, 55, 29, 47, 34 and 44 counts).

Besides those: real triggers (`Unknown`; see `integral-on-the-face.ipynb`), the six 20 % times exist and lie within 60 us of their median (one pulse), and the series `07221113_1159` (low in every channel) is left out. **No cut on the integral** is applied (the earlier notebooks required a total integral above 2, which would remove all of these small pulses).

**The x offset is -20 us, not Fritts's -1 us**: `xdel = t_E - (t_C + t_D) / 2 - 20 us`. Without it the distribution of these small events is centred at `xdel` of about 22 us, off the face; the 20 us only moves it back to the origin and is **not a derived or physical offset**. The y offset (+9 us) is Fritts's.

**The data** are the eight series of `r76-82v-dcrc1pb.dataset.toml`, reprocessed with the baseline window of samples 10 to 450. **The times** are those of `rise_crossing` (`fraction = 0.2`) on the raw traces, not filtered; for pulses this small (tens of counts against a noise of 2 to 3) they are noisy, so the distributions are broader than for large pulses. Run with the `darkmatter_cli_env` kernel.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow.compute as pc
import pyarrow.parquet as pq
from matplotlib import colors
from matplotlib.patches import Circle, Wedge

from goldpan.dataset import Dataset
from spots_82v import SPOTS

LETTERS = "ABCDEF"
_here = Path.cwd().resolve()
PROCESSING = next(p / "data_analysis" / "R76" / "processing" for p in [_here, *_here.parents] if (p / "data_analysis" / "R76" / "processing").is_dir())
OUT = Path(Dataset.open(PROCESSING / "r76-82v-dcrc1pb.dataset.toml").spec.output)
manifest = json.loads((OUT / "_goldpan.json").read_text(encoding="utf-8"))
PRE = manifest["streams"]["NAI-S2"]["detector"]["segments"][0]["pretrigger_samples"]
CACHE = Path.home() / "goldsluice-output" / f"relcal-positions-small-82v-p{PRE}.npz"
print(f"{OUT}: baseline window = samples 10 to {PRE}")

def field(tab, k, name):
    # one quantity of one channel: the channel is a struct column of the Parquet file
    return pc.struct_field(tab[f"Side2p{k}"].combine_chunks(), name).to_numpy(zero_copy_only=False)

if CACHE.exists():
    z = np.load(CACHE)
    series, trigger, bstd, height, t = z["series"], z["trigger"], z["bstd"], z["height"], z["t"]
else:
    S_, T_, B_, H_, X_ = [], [], [], [], []
    for f in sorted((OUT / "NAI-S2").glob("*_F[0-9][0-9][0-9][0-9].parquet")):
        tab = pq.read_table(f, columns=["series", "trigger_type"] + [f"Side2p{k}" for k in LETTERS])
        S_.append(np.asarray(tab["series"].to_pylist())); T_.append(np.asarray(tab["trigger_type"].to_pylist()))
        B_.append(np.column_stack([field(tab, k, "baseline_std") for k in LETTERS]))
        H_.append(np.column_stack([field(tab, k, "current_max") for k in LETTERS]))
        X_.append(np.column_stack([field(tab, k, "crossing_time_s") for k in LETTERS]) * 1e6)       # microseconds
    series, trigger, bstd, height, t = np.concatenate(S_), np.concatenate(T_), np.vstack(B_), np.vstack(H_), np.vstack(X_)
    np.savez(CACHE, series=series, trigger=trigger, bstd=bstd, height=height, t=t)
real, rand = trigger == "Unknown", trigger == "Physics"
print(f"{len(series):,} events in {len(np.unique(series))} series: {real.sum():,} real triggers")

## The cuts

The cut flow, step by step. The `bstd` cut is the dotted line of the `bstd` plots (twice the median of the random triggers of the channel), the same in every channel's own units.

In [ ]:
SATH = np.array([1277, 1095, 589, 930, 680, 883.0])         # saturated height of each channel (saturation-over-time.ipynb)
FRAC = 0.05
BSTD_CUT = np.array([2 * np.median(bstd[rand, i]) for i in range(6)])
WINDOW = 60.0                                               # microseconds: the six times of one pulse lie within this of their median
OFFSET_X, OFFSET_Y = -20.0, 9.0                             # Fritts's offsets are -1.0 us in x and +9.0 us in y; here x is -20 us: it puts the distribution of these small events near xdel = 0
EXCLUDE = {"07221113_1159"}
print("bstd cut (counts):      " + ", ".join(f"{k} {c:.2f}" for k, c in zip(LETTERS, BSTD_CUT)))
print("5 % of saturation:      " + ", ".join(f"{k} {FRAC * h:.0f}" for k, h in zip(LETTERS, SATH)))

keep = ~np.isin(series, list(EXCLUDE))
bstd_ok = (bstd < BSTD_CUT).all(axis=1)
small = (height < FRAC * SATH).all(axis=1)
finite = np.isfinite(t).all(axis=1)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    one_pulse = (np.abs(t - np.nanmedian(t, axis=1, keepdims=True)) < WINDOW).all(axis=1)
steps = [("real triggers", real & keep), ("+ bstd below the cut in all six channels", real & keep & bstd_ok),
         ("+ no channel above 5 % of its saturation", real & keep & bstd_ok & small),
         ("+ six 20 % times", real & keep & bstd_ok & small & finite),
         ("+ one pulse (times within 60 us)", real & keep & bstd_ok & small & finite & one_pulse)]
for name, m in steps:
    print(f"{name:<46}{m.sum():>11,}")
use = steps[-1][1]

xdel = t[:, 4] - 0.5 * (t[:, 2] + t[:, 3]) + OFFSET_X                 # E against the mean of C and D
ydel = 0.866 * (t[:, 3] - t[:, 2]) + OFFSET_Y                         # D against C
afdel = t[:, 5] - t[:, 0]                                             # F against A
phidel = np.degrees(np.arctan2(ydel, xdel))
for name, v in [("xdel", xdel), ("ydel", ydel), ("afdel", afdel)]:
    p = np.percentile(v[use], [1, 16, 50, 84, 99])
    print(f"{name:>6}  1/16/50/84/99 %: " + "  ".join(f"{x:7.2f}" for x in p) + "  (us)")

## The position distribution

On the left the plane of the delays, with the detector drawn under it as in the earlier notebooks (the `x` axis is reversed so that `C` is upper left, `D` lower left and `E` right; the scale `R_US` = 14 us for the radius of the face is a placeholder). The circles are the two high-activity spots found with the large events (`spots_82v.py`): spot C and spot F, 1.5 us radius. Then the azimuth `phidel` and the radius quantity `afdel`.

In [ ]:
REGIONS = {   # letter -> (inner radius, outer radius, start angle, end angle) in degrees, radius 1 = the face
    "A": (0.90, 1.00, 0, 360), "B": (0.80, 0.90, 0, 360),
    "C": (0.25, 0.80, 60, 180), "D": (0.25, 0.80, 180, 300), "E": (0.25, 0.80, -60, 60),
    "F": (0.00, 0.25, 0, 360),
}
R_US = 14.0       # PLACEHOLDER: the radius of the face in microseconds of delay
XR, YR = 22, 16

def draw_detector_mirrored(ax, radius=R_US, label=True, lw=0.8):
    for k, (r_in, r_out, a0, a1) in REGIONS.items():
        ax.add_patch(Wedge((0, 0), r_out * radius, 180 - a1, 180 - a0, width=(r_out - r_in) * radius,
                           facecolor="none", edgecolor="k", linewidth=lw, zorder=3))
        if label and k in "CDE":
            a = np.deg2rad((a0 + a1) / 2)
            ax.text(-0.52 * radius * np.cos(a), 0.52 * radius * np.sin(a), k, ha="center", va="center", fontsize=11, color="0.25", zorder=5)
    ax.set_aspect("equal")

def plane(ax, m, title, spots=False):
    h = ax.hist2d(xdel[m], ydel[m], bins=[160, 120], range=[[-XR, XR], [-YR, YR]], cmap="viridis", norm=colors.LogNorm(), cmin=1, zorder=2)
    draw_detector_mirrored(ax)
    if spots:
        for name, (x0, y0, r) in SPOTS.items():
            ax.add_patch(Circle((x0, y0), r, fill=False, ec="r", lw=0.8, ls="--", zorder=4))
            ax.text(x0, y0 + r + 0.5, name, color="r", ha="center", fontsize=9, zorder=5)
    ax.set_xlim(XR, -XR); ax.set_ylim(-YR, YR)
    ax.set_xlabel("xdel (us, reversed)"); ax.set_ylabel("ydel (us)")
    ax.set_title(f"{title}\n{m.sum():,} events", fontsize=10)
    return h

fig, axes = plt.subplots(1, 3, figsize=(18, 5.6), gridspec_kw={"width_ratios": [1.3, 1, 1]})
h = plane(axes[0], use, "bstd cut on all channels, no channel above 5 % of saturation", spots=True)
fig.colorbar(h[3], ax=axes[0], label="events per bin")
axes[1].hist(phidel[use], bins=np.linspace(-180, 180, 145), histtype="step", color="C0")
axes[1].set_xlabel("phidel = atan2(ydel, xdel) (degrees)"); axes[1].set_ylabel("events per bin"); axes[1].set_title("azimuth")
axes[2].hist(afdel[use], bins=np.linspace(-25, 25, 150), histtype="step", color="C0")
axes[2].set_xlabel("afdel = t_F - t_A (us)"); axes[2].set_ylabel("events per bin"); axes[2].set_title("radius quantity")
fig.tight_layout()
plt.show()

## What the cuts do to the distribution

The same plane for the events with only the `bstd` cut on all channels, and for the events with only the cut on the size, next to the selection of this notebook, so the effect of each cut can be seen. All three use the same conditions on the times (six 20 % times within 60 us of their median) and real triggers.

In [ ]:
base = real & keep & finite & one_pulse
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
plane(axes[0], base & bstd_ok, "bstd cut only (all channels)")
plane(axes[1], base & small, "no channel above 5 % of saturation only")
plane(axes[2], use, "both cuts")
fig.tight_layout()
plt.show()

## Only events with a visible pulse in every channel

Small, quiet events include some in which a channel has **no pulse at all** (the trigger is the phonon channel Side2pF, so the others may be noise); the 20 % time of such a channel is a time of the noise. The same selection with the extra condition that every channel has a pulse of at least 6 times its `bstd` (height / `bstd` above 6 in all six) shows how much that matters.

In [ ]:
visible = (height > 6 * bstd).all(axis=1)
use_v = use & visible
print(f"events with a pulse above 6 x bstd in every channel: {use_v.sum():,} of {use.sum():,} ({use_v.sum() / use.sum():.1%})")
print(f"{'':>10}" + "".join(f"{'Side2p' + k:>10}" for k in LETTERS))
print(f"{'visible':>10}" + "".join(f"{(height[use, i] > 6 * bstd[use, i]).mean():>10.1%}" for i in range(6)))
fig, axes = plt.subplots(1, 3, figsize=(18, 5.4), gridspec_kw={"width_ratios": [1.3, 1, 1]})
h = plane(axes[0], use_v, "the same, a pulse above 6 x bstd in every channel", spots=True)
fig.colorbar(h[3], ax=axes[0], label="events per bin")
axes[1].hist(phidel[use_v], bins=np.linspace(-180, 180, 145), histtype="step", color="C1")
axes[1].set_xlabel("phidel (degrees)"); axes[1].set_ylabel("events per bin"); axes[1].set_title("azimuth")
axes[2].hist(afdel[use_v], bins=np.linspace(-25, 25, 150), histtype="step", color="C1")
axes[2].set_xlabel("afdel (us)"); axes[2].set_ylabel("events per bin"); axes[2].set_title("radius quantity")
fig.tight_layout()
plt.show()

## A scan in the maximum pulse height, from about 0 to 30 %

The same plots, as a movie, with the cut on the size not fixed at 5 % of the saturated height but **scanned**: in every frame, no channel has a pulse height above a fraction `f` of its saturated height, with `f` from 0.5 % to 30 % (0.5, 1, 1.5, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 25 and 30 %). The other conditions are those of the notebook (real triggers, `bstd` below the cut in every channel, the six 20 % times within 60 us of their median, the x offset of -20 us). The selections are nested: an event in one frame is in all later ones. The colour scale is the same in all frames (events per bin), so the density grows from frame to frame; the two histograms have fixed axes. Saved as `figures/position-small-vs-height-max.gif`.

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image

FRACS = [0.005, 0.01, 0.015, 0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30]
common = real & keep & bstd_ok & finite & one_pulse
sets = [common & (height < f * SATH).all(axis=1) for f in FRACS]
print(f"{'f':>6} {'events':>9} {'xdel median':>12} {'ydel median':>12}")
for f, m in zip(FRACS, sets):
    med = (np.median(xdel[m]), np.median(ydel[m])) if m.sum() else (np.nan, np.nan)
    print(f"{f:>6.1%} {m.sum():>9,} {med[0]:>12.2f} {med[1]:>12.2f}")

xe = np.linspace(-XR, XR, 161); ye = np.linspace(-YR, YR, 121)
counts = [np.histogram2d(xdel[m], ydel[m], bins=[xe, ye])[0] for m in sets]
vmax = max(c.max() for c in counts)
pbins = np.linspace(-180, 180, 73); abins = np.linspace(-25, 25, 76)
pmax = max(np.histogram(phidel[m], bins=pbins)[0].max() for m in sets)
amax = max(np.histogram(afdel[m], bins=abins)[0].max() for m in sets)

fig, axes = plt.subplots(1, 3, figsize=(17, 5.2), gridspec_kw={"width_ratios": [1.3, 1, 1]})
def frame(k):
    f, m, c = FRACS[k], sets[k], counts[k]
    for ax in axes:
        ax.clear()
    ax = axes[0]
    ax.pcolormesh(xe, ye, np.where(c > 0, c, np.nan).T, cmap="viridis", norm=colors.LogNorm(vmin=1, vmax=vmax), zorder=2)
    draw_detector_mirrored(ax)
    ax.set_xlim(XR, -XR); ax.set_ylim(-YR, YR)
    ax.set_xlabel("xdel (us, reversed)"); ax.set_ylabel("ydel (us)")
    ax.set_title(f"no channel above {f:.1%} of its saturated height: {m.sum():,} events", fontsize=10)
    axes[1].hist(phidel[m], bins=pbins, histtype="step", color="C0")
    axes[1].set_xlim(-180, 180); axes[1].set_ylim(0, 1.05 * pmax)
    axes[1].set_xlabel("phidel (degrees)"); axes[1].set_ylabel("events per bin"); axes[1].set_title("azimuth")
    axes[2].hist(afdel[m], bins=abins, histtype="step", color="C0")
    axes[2].set_xlim(-25, 25); axes[2].set_ylim(0, 1.05 * amax)
    axes[2].set_xlabel("afdel (us)"); axes[2].set_ylabel("events per bin"); axes[2].set_title("radius quantity")
    return []

fig.tight_layout()
anim = FuncAnimation(fig, frame, frames=len(FRACS), interval=900, blit=False)
FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(exist_ok=True)
GIF = FIG_DIR / "position-small-vs-height-max.gif"
anim.save(GIF, writer=PillowWriter(fps=1.1), dpi=70)
plt.close(fig)
Image(filename=str(GIF))

### The same scan, zoomed way out

The same 15 selections and the same colour scale, but the plane of the delays drawn over +-80 us in `xdel` and +-60 us in `ydel` (the face is the small drawing in the middle), to see where the events that lie far from it go as the cut on the height is raised. Saved as `figures/position-small-vs-height-max-wide.gif`. The counts printed are the events inside this wider view.

In [ ]:
XW, YW = 80, 60
xw = np.linspace(-XW, XW, 201); yw = np.linspace(-YW, YW, 151)
wide = [np.histogram2d(xdel[m], ydel[m], bins=[xw, yw])[0] for m in sets]
vmax_w = max(c.max() for c in wide)
print(f"{'f':>6} {'in the view':>12} {'of':>9}")
for f, m, c in zip(FRACS, sets, wide):
    print(f"{f:>6.1%} {int(c.sum()):>12,} {m.sum():>9,}")

fig, ax = plt.subplots(figsize=(11, 8.6))
def frame_wide(k):
    f, m, c = FRACS[k], sets[k], wide[k]
    ax.clear()
    ax.pcolormesh(xw, yw, np.where(c > 0, c, np.nan).T, cmap="viridis", norm=colors.LogNorm(vmin=1, vmax=vmax_w), zorder=2)
    draw_detector_mirrored(ax, lw=1.2)
    ax.set_xlim(XW, -XW); ax.set_ylim(-YW, YW)
    ax.set_xlabel("xdel (us, reversed)"); ax.set_ylabel("ydel (us)")
    ax.set_title(f"no channel above {f:.1%} of its saturated height: {m.sum():,} events ({int(c.sum()):,} in the view)", fontsize=11)
    ax.grid(alpha=0.2)
    return []

fig.tight_layout()
anim_w = FuncAnimation(fig, frame_wide, frames=len(FRACS), interval=900, blit=False)
GIF_W = FIG_DIR / "position-small-vs-height-max-wide.gif"
anim_w.save(GIF_W, writer=PillowWriter(fps=1.1), dpi=70)
plt.close(fig)
Image(filename=str(GIF_W))

**What this shows, and what is not done.** The position distribution, from the delays, of the events that are quiet in every channel and small in every channel. Not done: millimetres (`R_US` is a placeholder), the offsets (the y offset is Fritts's, the x offset of -20 us is a shift to put the distribution near the origin; neither is derived), filtered times (the 20 % times are of the raw traces and are noisy for pulses of tens of counts), and a check of whether the two high-activity spots of the large events are the same objects as any feature here.